# Lesson 02 - Microsoft Agent Framework ကို ရှာဖွေခြင်း

**Microsoft Agent Framework (MAF)** သည် AI ကိုယ်စားလှယ်များ တည်ဆောက်ရန် အမျိုးတူ စနစ်တစ်ခုဖြစ်သည်။ ၎င်းသည် သန့်ရှင်း၍ ပေါင်းစပ်နိုင်သော structural ကို အာရုံစိုက်ပြီး အဓိက အဆောက်အအုံ ကဏ္ဍလေးခုပါရှိသည်-

- **Client** – AI model endpoint နှင့် ဆက်သွယ်ပြီး ဆက်သွယ်မှုကို ကိုင်တွယ်သည်။
- **Agent** – client ကို ညွှန်ကြားချက်များနှင့် ကိရိယာ သတ်မှတ်ချက်များဖြင့် အထုပ်ဖုံးသည်။
- **Tools** – မော်ဒယ်ခေါ်နိုင်သော စိတ်ကြိုက် လုပ်ဆောင်ချက်များဖြင့် ကိုယ်စားလှယ်စွမ်းဆောင်ရည်ကို တိုးချဲ့သည်။
- **Session** – မျိုးစုံ ဆက်သွယ်မှုများအတွက် စကားပြောမှတ်တမ်းကို ထိန်းသိမ်းသည်။

ဤသင်ခန်းစာတွင် ကျွန်ုပ်တို့သည် ဤအယူအဆများကို အသုံးပြု၍ ဒေသသတ်မှတ်ထားသော သွားရောက်ကြည့်ရှုခြင်းအတွက် **ခရီးသွားမှာယူခြင်း ကိုယ်စားလှယ်** တစ်ယောက်ထုတ်လုပ်မည်ဖြစ်သည်။


## စက်တင်််## Setup


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Agent Framework အဆောက်အအုံကိုနားလည်ခြင်း

Microsoft Agent Framework သည် အလွှာစီစဉ်မှုတစ်ရပ်ကိုလိုက်နာသည်။

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – `AzureAIProjectAgentProvider` သည် Azure OpenAI deployment တစ်ခုနှင့်ဆက်သွယ်သည်။ ၎င်းသည် authentication၊ request ဖွဲ့စည်းခြင်းနှင့် response ဖော်ပြခြင်းကို ကိုင်တွယ်ပေးသည်။
2. **Agent** – client မှ `provider.create_agent()` ဖြင့်တည်ဆောက်ပြီး မော်ဒယ်အသုံးပြုခြင်း၊ system prompt နှင့် tools များကိုပေါင်းစပ်ထားသည်။
3. **Tools** – `@tool` ဖြင့်တင်ဆက်ထားသော Python function များဖြစ်ပြီး agent က အမှုဆောင်ရန် သို့မဟုတ် မှတ်တမ်းယူရန် ဖော်ဆောင်နိုင်သည်။
4. **Session** – `AgentSession` object (agent.create_session() မှတည်ဆောက်) သည် စကားဝိုင်းမှတ်တမ်းများကိုသိမ်းဆည်းပြီး multi-turn ဆက်သွယ်မှု ရှိစေသည်။ Agent သည် ယခင်အကြောင်းအရာကိုမှတ်မိနိုင်သည်။

အလွှာတိုင်းကို တစ်ဆင့်ချင်းစီ တည်ဆောက်ကြပါစို့။


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool ဒီခေါင်းစဉ်ဖြင့် ကိရိယာများ ထည့်သွင်းခြင်း

ကိရိယာများသည် အေးဂျင့်များအား စာသားထုတ်ပေးခြင်းအပြင် အရေးယူနိုင်စေသည်။ `@tool` decorator သည် ပုံမှန် Python function ကို အေးဂျင့်ခေါ်ယူနိုင်သော အရာတစ်ခုအဖြစ် ပြောင်းလဲပေးသည်။

အဓိကအချက်များ
- မော်ဒယ်အား parameter တစ်ခုချင်းစီကို နားလည်ရန် `Annotated[type, "description"]` ကို အသုံးပြုပါ။
- docstring သည် မော်ဒယ်ကြည့်ရသောကိရိယာဖော်ပြချက် ဖြစ်သည်။
- `approval_mode="never_require"` ဆိုသည်မှာ အသုံးပြုသူ၏ အတည်ပြုချက်မလိုဘဲ ကိရိယာကို အလိုအလျောက် လည်ပတ်စေသည်။


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## ကိရိယာများသုံးပြီး Agent တစ်ခုဖန်တီးခြင်း

ယခု client၊ နှင့်လမ်းညွှန်ချက်များနှင့် ကိရိယာများကို agent တစ်ခုအဖြစ်ပေါင်းစပ်ပါမည်။ `instructions` သည် system prompt အဖြစ်လူထုဆောင်ပေးသည် — ၎င်းသည် agent ၏အပြုအမှုနှင့်နည်းလမ်းကိုသတ်မှတ်ပေးသည်။


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Multi-Turn Conversations with Sessions

An `AgentSession` (created via `agent.create_session()`) က အပြောအဆိုသမိုင်းလုံးကို မှတ်ထားပေးတယ်။ တူညီတဲ့ session ကို `agent.run()` ခေါ်တိုင်း ပေးလိုက်ခြင်းအားဖြင့် agent က စကားပြောသမိုင်းလုံးကို ကြည့်ရှုနိုင်ပြီး အကြောင်းအရင်းစာသားများကို ပြန်လည်ရည်ညွှန်းနိုင်သည်။

`tools=[check_destination_availability]` ကို ဖြတ်၍ agent သည် တစ်ခုချင်းစီ အဆုတ်အတောက်အတွင်း ကျွန်ုပ်တို့ရဲ့ရရှိနိုင်မှု စစ်ဆေးစက်ကို ခေါ်နိုင်သည်။


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## အကျဉ်းချုပ်

ဤစာသင်ခန်းတွင် Microsoft Agent Framework ၏ အဓိကခြေထောက်လေးချက်ကို သင်လေ့လာခဲ့သည်-

| ယူဆချက် | သင်လေ့လာသင်ယူခဲ့သည့်အရာ |
|---------|------------------|
| **Client** | `AzureAIProjectAgentProvider` သည် credential-based auth ဖြင့် Azure OpenAI နှင့် ချိတ်ဆက်သည် |
| **Agent** | `provider.create_agent()` သည် မော်ဒယ်ချိတ်ဆက်မှုကို ညွှန်ကြားချက်များနှင့် နာမည်တစ်ခုနှင့် ပေါင်းစပ်သည် |
| **Tools** | `@tool` decorator သည် agent မှခေါ်ယူနိုင်ရန် Python functions များကို ဖွင့်လှစ်ပေးသည် |
| **Session** | `agent.create_session()` သည် ကန့်သတ်ချက်များစွာ ကျော်လွန်သော စကားပြောမှုသမိုင်းကြောင်းကို ထိန်းသိမ်းသည် |

ဤဖွဲ့စည်းမှုအခြေခံများသည် သဘာဝစကားပြောနိုင်သော agent များ၊ ပြင်ပ function များကို ခေါ်ယူနိုင်သော၊ နှင့် သဘောတရားကို ထိန်းသိမ်းနိုင်သော agent များကို ဖန်တီးရန် ပေါင်းစည်းပေးလျက်ရှိသည်- နောက်ဆုံးတွင် ပိုမိုတိုးတက်သော agentic နမူနာများအတွက် အခြေခံလမ်းစဉ်ဖြစ်သည်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**တားမြစ်ချက်**  
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှုအတွက် ကြိုးစားရသော်လည်း၊ အလိုအလျောက် ဘာသာပြန်ခြင်းတွင် အမှားများ သို့မဟုတ် တိကျမှုဆိုင်ရာ မမှန်ကန်မှုများ ဖြစ်ပေါ်လာနိုင်ပါသည်။ မူရင်းစာတမ်းကို ရိုးရာဘာသာဖြင့်သာ တရားဝင် အရင်းအမြစ်အဖြစ် ယူဆရန် လိုအပ်ပါသည်။ အရေးကြီးသော အချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူဘာသာပြန်ကူညီမှု ရယူရန် အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းဒေါ်မှ ဖြစ်ပေါ်လာနိုင်သော အဓိပ္ပာယ်မှားမှုများ၊ ညွှန်ချွန်မှုမှားများအတွက် ကျွန်ုပ်တို့ တာဝန်မယူပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
